In [948]:
%cd /content
import os
if not os.path.exists("valley-outfitters-analysis"):
    !git clone https://github.com/LSRajan/valley-outfitters-analysis.git
%cd valley-outfitters-analysis

import pandas as pd
pd.set_option("display.max_columns", None)

/content
/content/valley-outfitters-analysis


In [949]:
orders_raw = pd.read_csv("data/raw/orders.csv")
customers_raw = pd.read_csv("data/raw/customers.csv")
order_items_raw = pd.read_csv("data/raw/order_items.csv")
products_raw = pd.read_csv("data/raw/products.csv")
returns_raw = pd.read_csv("data/raw/returns.csv")
web_sessions_raw = pd.read_csv("data/raw/web_sessions.csv")

In [950]:
def clean_labels(col, mapping, allowed):
  col = col.str.strip().str.lower().replace(mapping)
  print(col[~col.isin(allowed) & col.notna()])
  assert len(col[~col.isin(allowed) & col.notna()]) == 0
  return col


#1. Orders

In [951]:
orders = orders_raw.copy()

###Fix 01 : Duplicate rows in orders table are dropped

In [952]:
orders = orders.drop_duplicates().copy()

### Fix 03: Capitalise all customer ids

In [953]:
orders["customer_id"] = orders["customer_id"].str.upper()

###Fix 06: DATES TO DO________________________

###Fix 07: DATES TO DO________________________

###Fix 08: Standardise and relabel status labels

In [954]:
status_map = {
    "canceled" : "cancelled",
    "cancled" : "cancelled",
    "return" : "returned",
    "deliverd" : "delivered",
    "complete" : "delivered",
    "completed" : "delivered",
    "in progress" : "processing",
    "in transit" : "shipped"
}
status_allowed = ["delivered", "cancelled", "returned", "shipped", "pending", "processing"]

orders["status"] = clean_labels(orders["status"], status_map, status_allowed)

Series([], Name: status, dtype: object)


In [955]:
orders["payment_method"].str.strip().str.lower().value_counts()

,count
payment_method,
klarna,4933
paypal,4729
apple pay,4527
gift card,4490
card,4456
giftcard,506
applepay,501
pp,310
debit card,292


###Fix 09: Standardise and relabel payment method labels

In [956]:
payment_map = {
    "credit card" : "card",
    "debit card" : "card",
    "pp" : "paypal",
    "applepay" : "apple pay",
    "giftcard" : "gift card"
}
payment_allowed = ["card", "klarna", "apple pay", "gift card", "paypal"]

orders["payment_method"] = clean_labels(orders["payment_method"], payment_map, payment_allowed)

Series([], Name: payment_method, dtype: object)


###Fix 10: Standardise channel labels

In [957]:
orders["channel"] = orders["channel"].str.lower().str.strip()

###Fix 11: Standardise discount codes

In [958]:
orders["discount_code"] = orders["discount_code"].str.strip().str.lower()

###Fix 12 & Fix 13: Remove "£" prefix from order total. Take absolute value to account for negative values.

In [959]:
orders["order_total"] = orders["order_total"].str.replace("£", "").astype(float).abs()

#2. Customers

In [960]:
customers = customers_raw.copy()

###Fix 14: strip and lower first and last names.

In [961]:
customers["first_name"] = customers["first_name"].str.strip().str.lower()
customers["last_name"] = customers["last_name"].str.strip().str.lower()

###Fix 15 TO DO_____ NEED SIGN UP DATE CLEANED FIRST

In [962]:
customers["email"] = customers["email"].str.strip().str.lower()
customers.loc[customers[["email", "date_of_birth"]].duplicated(keep=False)
& customers["email"].notna()
& customers["date_of_birth"].notna()].sort_values(["email", "signup_date"], ascending=False)

,customer_id,first_name,last_name,email,phone,signup_date,date_of_birth,city,postcode,country,marketing_opt_in
1576,C6096,zainab,thomas,zainab.thomas70@yahoo.co.uk,07319 881663,unknown,1968-06-16,Bristol,BS18 6FJ,GB,TRUE
3774,C5270,zainab,thomas,zainab.thomas70@yahoo.co.uk,07319 881663,2026-03-31,1968-06-16,Bristol,BS4 8DD,United Kingdom,N
1877,C1607,tomasz,walker,tomasz.walker@hotmail.co.uk,07555 453771,2025-09-04,1955-01-18,Swansea,SA2 2WR,UK,TRUE
3797,C6276,tomasz,walker,tomasz.walker@hotmail.co.uk,07555 453771,14.10.25,1955-01-18,Swansea,SA16 1LA,United Kingdom,Y
4718,C6298,tomasz,walker,tomasz.walker72@icloud.com,+44 7974 271221,2024-02-12,09-05-1969,Bristol,BS3 3EE,UK,no
...,...,...,...,...,...,...,...,...,...,...,...
3784,C6134,aisha,thompson,aisha.thompson@gmail.com,07431 970680,09.05.25,1967-03-25,Belfast,BT11 3DT,UK,yes
4986,C6120,aisha,nowak,aisha.nowak@icloud.com,07904 660506,2025-12-31,2004-03-25,Newport,NP13 3YF,United Kingdom,NaN
1368,C4410,aisha,nowak,aisha.nowak@icloud.com,07904 660506,12-25-2025,2004-03-25,Newport,NP12 1RU,GB,No
2417,C3507,aisha,jones,aisha.jones12@hotmail.co.uk,0773760,2025-11-12,1993-08-15,Swansea,SA8 4ZX,UK,FALSE


###Fix 16 CLEAN SIGN UP DATE

###Fix 17 CLEAN DATE OF BIRTH

###Fix 18: Standardise and relabel city values

In [963]:
cities_map = {
    "cardif" : "cardiff",
    "londn" : "london",
    "manchestor": "manchester",
    "swanzea" : "swansea",
    "edinburg" : "edinburgh",
    "birmingam" : "birmingham",
}
cities_allowed = ["bristol","aberystwyth","sheffield","swansea","newport","glasgow","bath","edinburgh","exeter","bangor",
"nottingham","birmingham","leeds","cardiff","belfast","liverpool","wrexham","london","port talbot","manchester"]

customers["city"] = clean_labels(customers["city"], cities_map, cities_allowed)

Series([], Name: city, dtype: object)


###Fix 19: Relabel country labels

In [964]:
#country can be inferred from both city name and postcode when a city exists in multiple countries
countries_map = {
    "bristol": "england",
    "sheffield": "england",
    "bath": "england",
    "exeter": "england",
    "nottingham": "england",
    "birmingham": "england",
    "leeds": "england",
    "liverpool": "england",
    "london": "england",
    "manchester": "england",

    "aberystwyth": "wales",
    "swansea": "wales",
    "newport": "wales",
    "bangor": "wales",
    "cardiff": "wales",
    "wrexham": "wales",
    "port talbot": "wales",

    "glasgow": "scotland",
    "edinburgh": "scotland",

    "belfast": "northern ireland"
}
countries_allowed = ["england", "wales", "scotland", "northern ireland"]
customers["country"] = customers["city"].map(countries_map)
assert len(customers[~customers["country"].isin(countries_allowed) & customers["country"].notna()]) == 0

In [965]:
customers["marketing_opt_in"].str.strip().str.lower().value_counts(dropna=False)

,count
marketing_opt_in,
no,922
y,890
yes,867
n,470
0,460
false,437
1,436
NaN,431
true,407


###Fix 20: Standardise and relabel marketing opt in labels

In [966]:
marketing_map = {
    "n" : False,
    "no" : False,
    "0" : False,
    "false" : False,
    "y" : True,
    "yes" : True,
    "1" : True,
    "true" : True
}
marketing_allowed = [True, False]
customers["marketing_opt_in"] = clean_labels(customers["marketing_opt_in"], marketing_map, marketing_allowed)

Series([], Name: marketing_opt_in, dtype: object)


#3. Order Items

In [967]:
order_items = order_items_raw.copy()

###Fix 21: Drop the 250 duplicated rows

In [968]:
order_items = order_items.drop_duplicates().copy()
order_items.duplicated().sum()

np.int64(0)

##Fix 22: Capitalise product id string

In [969]:
order_items["product_id"] = order_items["product_id"].str.upper()

##Fix 24: Convert quantities to integer and take absolute value.

In [970]:
order_items["quantity"] = order_items["quantity"].replace({"one":1, "two":2, "three":3}).astype(int).abs()

##Fix 25: Replace outlier values with NA

In [971]:
order_items["quantity"] = order_items["quantity"].replace({100: pd.NA, 250: pd.NA, 999: pd.NA})

##Fix 26: Remove "£" prefix from unit price

In [972]:
order_items["unit_price"] = order_items["unit_price"].str.replace("£", "").astype(float)

#4. Products

In [973]:
products = products_raw.copy()

##Fix 27.0: Removed duplicated row in products

In [974]:
products = products.drop_duplicates().copy()

##Fix 27.1: Stripped and lowered name values

In [975]:
products["product_name"] = products["product_name"].str.strip().str.lower()

##Fix 27.2: Standardise and relabel category names

In [976]:
category_map = {
    "foot wear" : "footwear",
    "outer wear": "outerwear",
    "baselayers" : "base layers",
    "back packs" : "backpacks",
    "accesories" : "accessories",
    "camping & hiking" : "camping"
}
category_allowed = ["camping", "outerwear", "accessories", "base layers", "footwear", "backpacks"]
products["category"] = clean_labels(products["category"], category_map, category_allowed)

Series([], Name: category, dtype: object)


##Fix 28: Remove "£" and fix "," symbol from unit price

In [977]:
products["unit_price"] = products["unit_price"].str.replace("£", "").str.replace(",", ".").astype(float)

##Fix 29: Take absolute value of stock quantity

In [978]:
products["stock_qty"] = products["stock_qty"].abs()

##Fix 30: Standardise and relabel supplier names

In [979]:
supplier_map = {
    "northern gear ltd" : "northern gear",
    "blue ridge textiles" : "blueridge textiles",
    "blueridge textiles ltd" : "blueridge textiles",
    "trekco limited" : "trekco ltd",
    "alpine supply co" : "alpine supply company",
    "alpine supply co." : "alpine supply company",
    "summit wholesale ltd" : "summit wholesale"
}
supplier_allowed = ["northern gear", "blueridge textiles", "trekco ltd", "alpine supply company", "summit wholesale"]
products["supplier"] = clean_labels(products["supplier"], supplier_map, supplier_allowed)

Series([], Name: supplier, dtype: object)


##FIX 31 DATE CLEANING

##Fix 32: standardise and relabel is active values

In [980]:
products["is_active"].str.lower().str.strip().value_counts()

,count
is_active,
n,39
true,31
0,25
false,24
1,22
y,22


In [981]:
active_map = {
    "n" : False,
    "no" : False,
    "0" : False,
    "false" : False,
    "y" : True,
    "yes" : True,
    "1" : True,
    "true" : True
}
active_allowed = [True, False]
products["is_active"] = clean_labels(products["is_active"], active_map, active_allowed)

Series([], Name: is_active, dtype: bool)


/tmp/ipykernel_1855/439179522.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  col = col.str.strip().str.lower().replace(mapping)


#5. Returns

In [982]:
returns = returns_raw.copy()

##FIX 33 CLEAN DATE COLUMN

##Fix 34: Standardise and relabel return reasons

In [983]:
reason_map = {
    "-" : pd.NA,
    "see email" : pd.NA,
    "too small" : "size",
    "didn't fit" : "size",
    "wrong size" : "size",
    "didnt fit" : "size",
    "doesn't fit" : "size",
    "too big" : "size",

    "found cheaper" : "changed mind",
    "no longer needed" : "changed mind",

    "box crushed" : "faulty delivery",
    "damaged" : "faulty delivery",
    "damaged in transit" : "faulty delivery",
    "arrived damaged" : "faulty delivery",
    "faulty zip" : "faulty product",

    "stitching came apart" : "faulty product",
    "not waterproof" : "faulty product",
    "zip broke" : "faulty product",
    "faulty" : "faulty product",
    "leaking" : "faulty product",

    "colour different" : "misleading",
    "not as described" : "misleading",
    "looks different to photo" : "misleading",

    "bought by mistake" : "mistake",

}
reason_allowed = ["size", "changed mind", "faulty delivery", "faulty product", "mistake", "misleading", "other"]
returns["reason"] = clean_labels(returns["reason"], reason_map, reason_allowed)

Series([], Name: reason, dtype: object)


##Fix 35: Strip and lower resolution column

In [984]:
returns["resolution"] = returns["resolution"].str.strip().str.lower()

#6. Web Sessions

In [985]:
web_sessions = web_sessions_raw

##Fix 36.0 Drop duplicated rows

In [986]:
web_sessions = web_sessions.drop_duplicates().copy()

##Fix 37 CLEAN DATE TIME COLUMN

##Fix 38: Standardise and relabel device labels.

In [987]:
device_map = {
    "iphone" : "mobile",
    "pc" : "desktop",
    "ipad" : "tablet"
}
device_allowed = ["mobile", "desktop", "tablet"]
web_sessions["device"] = clean_labels(web_sessions["device"], device_map, device_allowed)

Series([], Name: device, dtype: object)


##Fix 39: Standardise and relabel traffice source values.

In [988]:
traffic_map = {
    "none" : "direct",
    "e-mail" : "email",
    "(direct)" : "direct",
    "paid_search" : "paid search",
    "instagram" : "social",
    "facebook" : "social",
    "fb" : "social",
    "newsletter" : "email",
    "google organic" : "organic",
    "organic search" : "organic",
    "referral site" : "referral",
    "google_ads" : "ads",
    "google ads" : "ads",
    "ppc" : "ads"
}
traffic_allowed = ["email", "direct", "paid search", "ads", "social", "organic", "referral"]
web_sessions["traffic_source"] = clean_labels(web_sessions["traffic_source"], traffic_map, traffic_allowed)

Series([], Name: traffic_source, dtype: object)


##Fix 40: Take absolute value of pages viewed.

In [989]:
web_sessions["pages_viewed"] = web_sessions["pages_viewed"].abs()

##Fix 41: Standardise and relabel converted column values

In [990]:
web_sessions["converted"].value_counts()

,count
converted,
0,48218
N,4920
FALSE,4751
1,1733
Y,206
TRUE,172


In [991]:
convert_map = {
    "0" : False,
    "n" : False,
    "false" : False,
    "1" : True,
    "y" : True,
    "true" : True
}
convert_allowed = [True, False]
web_sessions["converted"] = clean_labels(web_sessions["converted"], convert_map, convert_allowed)

Series([], Name: converted, dtype: bool)


/tmp/ipykernel_1855/439179522.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  col = col.str.strip().str.lower().replace(mapping)
